# 02_baseline — 自己リピート + 人気度ハイブリッド ベースライン

01_eda.ipynb で検証した仮説駆動EDA(`00_md/hypothesis_driven_design.md`)を踏まえたベースライン。

- 候補生成: 自己リピート(H1: article単位で14.1%が完全再購入)を最優先し、不足分を直近1週間の人気商品(H10: 死に筋を除いた"生きている"商品で人気度を測る)で埋める。
- コールドスタート顧客(H9: 学習期間の購入0〜1件が10.3%)は自動的に人気度リストのみで埋まる(自己リピート候補が空のため)。
- 検証は時系列バックテスト(直近1週間をホールドアウト)でMAP@12を測定してから、全データで最終候補を作り直して提出用csvを作成する。
- 目的: フル pipeline(候補生成→特徴量→LightGBM再ランキング)に入る前に、提出フォーマットの疎通確認と最初のPublic LBスコアを得ること。

In [ ]:
import os
import duckdb
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

DATA_DIR = "../01_input"
TRANS_LITE_PATH = os.path.join("..", "02_output", "features", "trans_lite.parquet")
SUBMISSION_DIR = os.path.join("..", "02_output", "submission")
EXPERIMENT_LOG_PATH = os.path.join("..", "experiment_log.csv")

con = duckdb.connect()
con.execute("PRAGMA threads=4")
con.execute(f"CREATE OR REPLACE VIEW trans AS SELECT * FROM read_parquet('{TRANS_LITE_PATH}')")
con.execute("SELECT COUNT(*) AS n_rows, MIN(t_dat) AS min_dat, MAX(t_dat) AS max_dat FROM trans").df()

In [ ]:
# trans_lite.parquet は customer_id を customer_idx(整数)に変換済みのため、
# 提出時に customer_id へ戻すための対応表を customers.csv から作り直す(01_eda.ipynbのcust_map定義と同一ロジック)
con.execute(f"""
CREATE OR REPLACE TABLE cust_map AS
SELECT customer_id, (ROW_NUMBER() OVER () - 1)::INTEGER AS customer_idx
FROM read_csv_auto('{DATA_DIR}/customers.csv')
""")
con.execute("SELECT COUNT(*) FROM cust_map").df()

## MAP@12 評価関数

コンペの評価指標(Average Precision@12 の顧客平均)をそのまま実装する。

In [ ]:
def apk(actual: set, predicted: list, k: int = 12) -> float:
    if not actual:
        return 0.0
    predicted = predicted[:k]
    hits = 0
    score = 0.0
    for i, p in enumerate(predicted):
        if p in actual:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(actual), k)


def mapk(actual_map: dict, pred_map: dict, k: int = 12) -> float:
    scores = []
    for customer_idx, actual in actual_map.items():
        predicted = pred_map.get(customer_idx, [])
        scores.append(apk(actual, predicted, k))
    return float(np.mean(scores)) if scores else 0.0

## 候補生成ロジック

`cutoff` 以前のデータのみを使い、
1. 自己リピート候補: 顧客ごとに過去購入した article_id を「直近購入日 → 購入回数」の順でランキングし上位12件(H1)
2. 人気度候補: cutoff 直前1週間に売れた article_id を件数順に12件(死に筋を除外する効果もある、H10)

の2つをこの順で結合し、重複を除いて12件に切り詰める。

In [ ]:
def build_repeat_candidates(con, cutoff: str) -> dict:
    df = con.execute(f"""
        WITH agg AS (
            SELECT customer_idx, article_id, COUNT(*) AS freq, MAX(t_dat) AS last_dat
            FROM trans
            WHERE t_dat <= DATE '{cutoff}'
            GROUP BY customer_idx, article_id
        ),
        ranked AS (
            SELECT customer_idx, article_id,
                   ROW_NUMBER() OVER (PARTITION BY customer_idx ORDER BY last_dat DESC, freq DESC) AS rn
            FROM agg
        )
        SELECT customer_idx, LIST(article_id ORDER BY rn) AS repeat_articles
        FROM ranked
        WHERE rn <= 12
        GROUP BY customer_idx
    """).df()
    return dict(zip(df.customer_idx, df.repeat_articles))


def build_popularity_candidates(con, cutoff: str, window_days: int = 7, top_n: int = 12) -> list:
    df = con.execute(f"""
        SELECT article_id, COUNT(*) AS cnt
        FROM trans
        WHERE t_dat <= DATE '{cutoff}' AND t_dat > DATE '{cutoff}' - INTERVAL {window_days} DAY
        GROUP BY article_id
        ORDER BY cnt DESC
        LIMIT {top_n}
    """).df()
    return df.article_id.tolist()


def build_predictions(customer_idx_list, repeat_map: dict, pop_list: list, k: int = 12) -> dict:
    preds = {}
    for customer_idx in customer_idx_list:
        candidates = list(repeat_map.get(customer_idx, []))
        if len(candidates) < k:
            seen = set(candidates)
            for a in pop_list:
                if a not in seen:
                    candidates.append(a)
                    seen.add(a)
                if len(candidates) >= k:
                    break
        preds[customer_idx] = candidates[:k]
    return preds

## バックテスト

学習データの最終日(2020-09-22)から直近7日間をホールドアウトし、それ以前のデータのみで候補を作って、
実際にホールドアウト期間に購入されたarticleに対するMAP@12を測定する。

In [ ]:
max_dat = con.execute("SELECT MAX(t_dat) FROM trans").fetchone()[0]
backtest_cutoff = (max_dat - timedelta(days=7)).strftime("%Y-%m-%d")
print(f"max_dat = {max_dat}, backtest_cutoff = {backtest_cutoff}")

actual_df = con.execute(f"""
    SELECT DISTINCT customer_idx, article_id
    FROM trans
    WHERE t_dat > DATE '{backtest_cutoff}'
""").df()
actual_map = actual_df.groupby("customer_idx")["article_id"].apply(set).to_dict()
print(f"バックテスト期間に購入があった顧客数: {len(actual_map):,}")

In [ ]:
repeat_map_bt = build_repeat_candidates(con, backtest_cutoff)
pop_list_bt = build_popularity_candidates(con, backtest_cutoff)

print(f"自己リピート候補を持つ顧客数: {len(repeat_map_bt):,}")
print(f"人気度フォールバック(直近1週間トップ12): {pop_list_bt}")

pred_map_bt = build_predictions(actual_map.keys(), repeat_map_bt, pop_list_bt)
backtest_score = mapk(actual_map, pred_map_bt)
print(f"バックテスト MAP@12 = {backtest_score:.4f}")

## 全データで最終候補を作成し、提出ファイルを生成

バックテストと同じロジックで、cutoffを学習データの最終日(2020-09-22)にして全顧客分の候補を作る。

In [ ]:
final_cutoff = max_dat.strftime("%Y-%m-%d")
repeat_map_final = build_repeat_candidates(con, final_cutoff)
pop_list_final = build_popularity_candidates(con, final_cutoff)

print(f"自己リピート候補を持つ顧客数: {len(repeat_map_final):,}")
print(f"人気度フォールバック(直近1週間トップ12): {pop_list_final}")

In [ ]:
all_customers = con.execute("SELECT customer_id, customer_idx FROM cust_map").df()
pred_map_final = build_predictions(all_customers.customer_idx, repeat_map_final, pop_list_final)

def format_prediction(article_ids):
    return " ".join(str(a).zfill(10) for a in article_ids)

all_customers["prediction"] = all_customers["customer_idx"].map(pred_map_final).apply(format_prediction)
submission = all_customers[["customer_id", "prediction"]]
submission.head()

In [ ]:
assert len(submission) == len(all_customers)
assert submission["prediction"].apply(lambda s: len(s.split())).eq(12).all(), "12件に満たない顧客がいます"

sample_submission = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"), usecols=["customer_id"])
assert set(submission["customer_id"]) == set(sample_submission["customer_id"]), "sample_submission.csvの顧客と一致しません"
print("フォーマット検証OK")

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
submission_filename = f"submission_{timestamp}_cv{backtest_score:.4f}.csv"
submission_path = os.path.join(SUBMISSION_DIR, submission_filename)
submission.to_csv(submission_path, index=False)
print(f"保存しました: {submission_path}")

## 実験ログへの記録

In [ ]:
log_row = pd.DataFrame([{
    "run_id": f"run_{timestamp}",
    "timestamp": timestamp,
    "notebook": "02_baseline.ipynb",
    "description": "自己リピート+人気度ハイブリッドのルールベースベースライン(候補生成のみ、モデルなし)",
    "features": "self_repeat(article_id, 直近購入日+頻度順), popularity(直近7日間の売上件数トップ12)",
    "params": "repeat_top_k=12, popularity_window_days=7, popularity_top_n=12",
    "cv_score": round(backtest_score, 4),
    "public_lb": "",
    "submission_file": submission_filename,
    "model_file": "",
    "notes": "バックテストは直近7日間ホールドアウト。Public LB判明後に追記すること。",
}])

if os.path.exists(EXPERIMENT_LOG_PATH):
    existing = pd.read_csv(EXPERIMENT_LOG_PATH)
    updated = pd.concat([existing, log_row], ignore_index=True)
else:
    updated = log_row

updated.to_csv(EXPERIMENT_LOG_PATH, index=False)
updated.tail()

## 次のステップ

- このベースラインをKaggleに提出し、Public LBスコアを`experiment_log.csv`に追記する。
- `03_feature_engineering.ipynb`: `hypothesis_driven_design.md` 4節の候補生成(セグメント別人気・チャネル考慮の共起・トレンド・価格帯フィルタ)とuser/item/user-item交差特徴量を実装する。
- `04_train_lgbm.ipynb`: LightGBM(binary or LambdaRank, group=customer_id)による再ランキングで、このベースラインを上回れるか検証する。